# Common mistakes mini-lesson

Each section pairs a mistake with a fix including runnable code and saved figures.

## Degrees vs radians
Plotting sin over degrees looks wrong; convert to radians before calling numpy trig functions.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

x_deg = np.linspace(0, 360, 361)
y_deg = np.sin(x_deg)
fig, ax = plt.subplots()
ax.plot(x_deg, y_deg, label='sin in degrees (wrong)')
ax.legend()
ax.set_title('Mistake: degrees instead of radians')
fig.savefig('assets/figures/common_mistakes_degrees.png', dpi=150, bbox_inches='tight')
plt.close(fig)

x_rad = np.linspace(0, 2 * np.pi, 361)
y_rad = np.sin(x_rad)
fig, ax = plt.subplots()
ax.plot(x_rad, y_rad, label='sin in radians (correct)')
ax.legend()
ax.set_title('Correct: radians')
fig.savefig('assets/figures/common_mistakes_radians.png', dpi=150, bbox_inches='tight')
plt.close(fig)


## Too-large range for exp
Applying np.exp to huge inputs overflows; clip or rescale the input.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

x = np.linspace(0, 1000, 400)
try:
    np.exp(x)
except OverflowError as exc:
    print('overflow:', exc)
y_safe = np.exp(np.clip(x, None, 50))
fig, ax = plt.subplots()
ax.plot(x, y_safe)
ax.set_title('Fix: clip before exp')
fig.savefig('assets/figures/common_mistakes_exp.png', dpi=150, bbox_inches='tight')
plt.close(fig)


## Poor step size h
Choosing h too small yields rounding error; opt for sqrt(eps) instead.

In [ ]:
import numpy as np
from calculus_with_python.differentiation import difference_quotient
import matplotlib.pyplot as plt

def target(x):
    return np.sin(x)

x0 = 1.0
h_values = np.logspace(-16, -1, 60)
errors = []
for h in h_values:
    try:
        approx = difference_quotient(target, x0, h=h, method='central')
        errors.append(abs(approx - np.cos(x0)))
    except ValueError:
        errors.append(np.nan)
fig, ax = plt.subplots()
ax.loglog(h_values, errors, marker='o')
ax.set_title('Error vs h (central difference)')
ax.set_xlabel('h')
ax.set_ylabel('abs error')
ax.grid(True, which='both', linestyle=':')
fig.savefig('assets/figures/common_mistakes_h.png', dpi=150, bbox_inches='tight')
plt.close(fig)


## Non-uniform grids vs Simpson
Simpson requires uniform spacing; mixing random grids breaks the assumption so trapezoid is safer.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from calculus_with_python.integration import integrate_f

ns = [51, 101, 151]
values = {method: [] for method in ['trapezoid', 'simpson']}
for method in values:
    for n in ns:
        try:
            val = integrate_f(np.sin, 0.0, 1.0, n=n, method=method)
        except ValueError:
            val = np.nan
        values[method].append(val)
fig, ax = plt.subplots()
for method, vals in values.items():
    ax.plot(ns, vals, marker='o', label=method)
ax.set_title('Non-uniform grids break Simpson')
ax.set_xlabel('n')
ax.set_ylabel('approx integral')
ax.legend()
fig.savefig('assets/figures/common_mistakes_simpson.png', dpi=150, bbox_inches='tight')
plt.close(fig)


## Domain issues
Log and sqrt require non-negative inputs; guard or clamp values before calling them.

In [ ]:
import numpy as np

def safe_log(x):
    if np.any(x <= 0):
        raise ValueError('x must be positive for log')
    return np.log(x)

xs = np.linspace(-1.0, 2.0, 30)
try:
    _ = safe_log(xs)
except ValueError as exc:
    print('domain error:', exc)
clamped = np.log(np.clip(xs, 1e-6, None))
print('clamped log sample:', clamped[:5])


## Plotting artifacts
Under-resolving high-frequency functions hides features; increase sampling density.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def sharp(x):
    return np.sin(50 * x)

x_sparse = np.linspace(0, 1, 20)
y_sparse = sharp(x_sparse)
fig, ax = plt.subplots()
ax.plot(x_sparse, y_sparse, marker='o')
ax.set_title('Under-resolved plot (20 points)')
fig.savefig('assets/figures/common_mistakes_plot_lowres.png', dpi=150, bbox_inches='tight')
plt.close(fig)

x_fine = np.linspace(0, 1, 400)
y_fine = sharp(x_fine)
fig, ax = plt.subplots()
ax.plot(x_fine, y_fine)
ax.set_title('Better resolution (400 points)')
fig.savefig('assets/figures/common_mistakes_plot_highres.png', dpi=150, bbox_inches='tight')
plt.close(fig)
